# 阶段 0：冻结基线

目标：确认**现在板子上到底是哪个 overlay、里面有哪些 IP**，并把这一刻存档。

这一步不写任何硬件，只是“体检 + 存档”。后面每个阶段出问题，都要靠这里的记录回退。

In [ ]:
import os
import sys
import json
import hashlib
import datetime

# 自动定位工程根目录：从当前目录往上找，直到看见 lib/video_setup.py。
# 这样 ai_video 无论放在 /home/xilinx/jupyter_notebooks/ai_video，
# 还是塞在 OV5640_capture_only/ai_video 里，notebook 都不用改路径。
ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT) and not os.path.isfile(os.path.join(ROOT, "lib", "video_setup.py")):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)

from lib import video_setup as vs

ol = vs.load_overlay(ROOT)

## 检查 1：基础 IP 是否齐全，自定义 IP 是否已经挂上

In [ ]:
ip_names = sorted(ol.ip_dict.keys())

BASIC = ["axi_iic_0", "axi_vdma_0", "axi_vdma_1", "v_demosaic_0", "v_tc_0", "rgb2dvi_0"]
missing_basic = [n for n in BASIC if n not in ip_names]

OUR_IP = "axis_video_proc_0"
has_our_ip = OUR_IP in ip_names

print("IP 总数:", len(ip_names))
print("缺少基础 IP:", missing_basic if missing_basic else "无")
print("自定义 IP %s 是否存在:" % OUR_IP, has_our_ip)

if has_our_ip:
    from lib.ai_video_regs import AiVideoRegs
    regs = AiVideoRegs(ol.axis_video_proc_0)
    print("  VERSION = 0x%08X" % regs.version())
    print("  当前 MODE_SEL =", regs.mode_sel())
    print("  FRAME_CNT   =", regs.frame_cnt())

## 检查 2：列出带 AXI-Lite 接口的 IP

后面阶段 2/6/7/8 要控制的全是这一类 IP。如果列表里没有你自己的 IP，
说明它还没被加上 AXI-Lite 从接口。

In [ ]:
axil = []
for name, entry in ol.ip_dict.items():
    if isinstance(entry, dict) and ("registers" in entry or "driver" in entry):
        axil.append(name)
print("带 AXI-Lite 的 IP (%d 个):" % len(axil))
for n in sorted(axil):
    print("  -", n)

## 检查 3：给出结论 + 存档

把 `.bit` / `.hwh` 的 SHA256 记到 `data/baseline.json`。
以后任何阶段改坏了，都可以用这份记录确认“改动前的版本长什么样”。

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

bit_path = vs.find_overlay(ROOT)
hwh_path = os.path.splitext(bit_path)[0] + ".hwh"

info = {
    "time": datetime.datetime.now().isoformat(timespec="seconds"),
    "bit": bit_path,
    "bit_sha256": sha256(bit_path),
    "hwh": hwh_path,
    "hwh_sha256": sha256(hwh_path) if os.path.exists(hwh_path) else None,
    "ip_names": ip_names,
    "has_axis_video_proc": has_our_ip,
    "missing_basic_ips": missing_basic,
}

data_dir = os.path.join(ROOT, "data")
os.makedirs(data_dir, exist_ok=True)
out_path = os.path.join(data_dir, "baseline.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(info, f, indent=2, ensure_ascii=False)

print("已保存:", out_path)
print(json.dumps(info, indent=2, ensure_ascii=False))

## 结论 → 下一步

* `自定义 IP 是否存在` 打印 **False**（预期情况）：
  进入 **阶段 1**，用 Vivado 把 `axis_video_proc` 直通版插进 BD。
* 打印 **True** 且 `VERSION = 0x00010000`：
  说明阶段 1~2 已完成，可直接跳到 **阶段 3**。
* `缺少基础 IP` 不是「无」：
  说明 overlay 放错了，回到 `README.md` 第 2 节核对目录。